### Libaraies

In [1]:
import requests
import json

### Step 1: Setup

In [2]:
BASE_URL = "http://localhost:8000"

def show(r):
    print(f"Status  : {r.status_code}")
    print(json.dumps(r.json(), indent=2))

print(" Setup complete")
print(f"   API URL : {BASE_URL}")

 Setup complete
   API URL : http://localhost:8000


### Step 2: Test /health

In [4]:
r = requests.get(f"{BASE_URL}/health")
print(f"Status : {r.status_code}")

data = r.json()
print(f"\nAPI Status : {data['status']}")
print(f"Version    : {data['version']}")
print(f"\nModels loaded:")
for model, loaded in data['models'].items():
    status = "Done" if loaded else "Nope"
    print(f"   {status} {model}")

Status : 200

API Status : healthy
Version    : 1.0.0

Models loaded:
   Done cls_onnx
   Done reg_onnx
   Done cluster_onnx
   Done cls_pipeline
   Done reg_pipeline
   Done cls_fe_pipeline
   Done reg_fe_pipeline
   Done cls_explainer
   Done reg_explainer
   Done database


### Step 3: Test /stats

In [5]:
r = requests.get(f"{BASE_URL}/stats")
print(f"Status : {r.status_code}")

data = r.json()
print(f"\nTotal Predictions : {data['total']}")
print(f"Approved          : {data['approved']}")
print(f"Rejected          : {data['rejected']}")
print(f"Avg Default Prob  : {data['avg_default_probability']}")
print(f"Avg Interest Rate : {data['avg_interest_rate']}")

Status : 200

Total Predictions : 14
Approved          : 7
Rejected          : 7
Avg Default Prob  : 0.5315
Avg Interest Rate : 15.5316


### Step 4: Test /predict/new

In [7]:
payload_rejected = {
    "cnic"                      : "42101-1234567-1",
    "loan_amnt"                 : 15000,
    "loan_grade"                : "F",
    "loan_intent"               : "DEBTCONSOLIDATION",
    "loan_percent_income"       : 0.43,
    "person_income"             : 35000,
    "person_age"                : 28,
    "person_emp_length"         : 2.0,
    "person_home_ownership"     : "RENT",
    "cb_person_default_on_file" : "N",
    "cb_person_cred_hist_length": 3.0
}

r = requests.post(f"{BASE_URL}/predict/new", json=payload_rejected)
print(f"Status : {r.status_code}")

data = r.json()
cls  = data['classification']
reg  = data['regression']
clu  = data['clustering']

print(f"\nWorkflow     : {data['workflow']}")
print(f"Prediction ID: {data['prediction_id']}")
print(f"\nDecision     : {cls['decision']}")
print(f"Default Prob : {cls['default_probability']}")
print(f"Predicted Rate: {reg['predicted_interest_rate']}%")
print(f"Cluster      : {clu['cluster_label']}")
print(f"\nTop reason   : {data['plain_reasons']['decision_reasons'][0]}")

Status : 200

Workflow     : new_applicant
Prediction ID: 19

Decision     : REJECTED
Default Prob : 1.0
Predicted Rate: 18.7842%
Cluster      : High Value Borrower

Top reason   : Your loan application was REJECTED with a 100.0% estimated default risk (threshold: 60%).


In [8]:
payload_approved = {
    "cnic"                      : "42101-9999999-9",
    "loan_amnt"                 : 5000,
    "loan_grade"                : "A",
    "loan_intent"               : "EDUCATION",
    "loan_percent_income"       : 0.08,
    "person_income"             : 65000,
    "person_age"                : 35,
    "person_emp_length"         : 8.0,
    "person_home_ownership"     : "MORTGAGE",
    "cb_person_default_on_file" : "N",
    "cb_person_cred_hist_length": 10.0
}

r = requests.post(f"{BASE_URL}/predict/new", json=payload_approved)
print(f"Status : {r.status_code}")

data = r.json()
cls  = data['classification']
reg  = data['regression']
clu  = data['clustering']

print(f"\nWorkflow      : {data['workflow']}")
print(f"Prediction ID : {data['prediction_id']}")
print(f"\nDecision      : {cls['decision']}")
print(f"Default Prob  : {cls['default_probability']}")
print(f"Predicted Rate: {reg['predicted_interest_rate']}%")
print(f"Cluster       : {clu['cluster_label']}")
print(f"\nTop reason    : {data['plain_reasons']['decision_reasons'][0]}")

Status : 200

Workflow      : new_applicant
Prediction ID : 20

Decision      : APPROVED
Default Prob  : 0.0953
Predicted Rate: 10.9955%
Cluster       : Standard Borrower

Top reason    : Your loan application was APPROVED with only a 9.5% estimated default risk (threshold: 60%).


### Step 5: Test /predict/existing

In [20]:
payload_existing_rejected = {
    "cnic"                      : "42101-1234567-1",
    "loan_amnt"                 : 15000,
    "loan_int_rate"             : 19.4,
    "loan_grade"                : "F",
    "loan_intent"               : "DEBTCONSOLIDATION",
    "loan_percent_income"       : 0.43,
    "person_income"             : 35000,
    "person_age"                : 28,
    "person_emp_length"         : 2.0,
    "person_home_ownership"     : "RENT",
    "cb_person_default_on_file" : "N",
    "cb_person_cred_hist_length": 3.0
}

r = requests.post(f"{BASE_URL}/predict/existing", json=payload_existing_rejected)
print(f"Status : {r.status_code}")

data = r.json()
cls  = data['classification']
clu  = data['clustering']

print(f"\nWorkflow      : {data['workflow']}")
print(f"Prediction ID : {data['prediction_id']}")
print(f"\nDecision      : {cls['decision']}")
print(f"Default Prob  : {cls['default_probability']}")
print(f"Regression    : Not run — user provided rate of 19.4%")
print(f"Cluster       : {clu['cluster_label']}")
print(f"\nTop reason    : {data['plain_reasons']['decision_reasons'][0]}")

Status : 200

Workflow      : existing_loan
Prediction ID : 30

Decision      : REJECTED
Default Prob  : 1.0
Regression    : Not run — user provided rate of 19.4%
Cluster       : High Value Borrower

Top reason    : Your loan application was REJECTED with a 100.0% estimated default risk (threshold: 60%).


In [21]:
payload_existing_approved = {
    "cnic"                      : "42101-9999999-9",
    "loan_amnt"                 : 5000,
    "loan_int_rate"             : 7.5,
    "loan_grade"                : "A",
    "loan_intent"               : "EDUCATION",
    "loan_percent_income"       : 0.08,
    "person_income"             : 65000,
    "person_age"                : 35,
    "person_emp_length"         : 8.0,
    "person_home_ownership"     : "MORTGAGE",
    "cb_person_default_on_file" : "N",
    "cb_person_cred_hist_length": 10.0
}

r = requests.post(f"{BASE_URL}/predict/existing", json=payload_existing_approved)
print(f"Status : {r.status_code}")

data = r.json()
cls  = data['classification']
clu  = data['clustering']

print(f"\nWorkflow      : {data['workflow']}")
print(f"Prediction ID : {data['prediction_id']}")
print(f"\nDecision      : {cls['decision']}")
print(f"Default Prob  : {cls['default_probability']}")
print(f"Regression    : Not run — user provided rate of 7.5%")
print(f"Cluster       : {clu['cluster_label']}")
print(f"\nTop reason    : {data['plain_reasons']['decision_reasons'][0]}")

Status : 200

Workflow      : existing_loan
Prediction ID : 31

Decision      : APPROVED
Default Prob  : 0.0099
Regression    : Not run — user provided rate of 7.5%
Cluster       : Standard Borrower

Top reason    : Your loan application was APPROVED with only a 1.0% estimated default risk (threshold: 60%).


### Step 6: Test GET /applicant/{cnic}

In [23]:
CNIC = "42101-1234567-1"

r = requests.get(f"{BASE_URL}/applicant/{CNIC}")
print(f"Status : {r.status_code}")

data = r.json()
print(f"\nCNIC           : {data['cnic']}")
print(f"First Seen     : {data['first_seen']}")
print(f"Last Seen      : {data['last_seen']}")
print(f"Total Visits   : {data['total_visits']}")
print(f"Total Approved : {data['total_approved']}")
print(f"Total Rejected : {data['total_rejected']}")
print(f"Last Decision  : {data['last_decision']}")
print(f"Last Loan Amnt : {data['last_loan_amnt']}")
print(f"Last Rate      : {data['last_interest_rate']}")

lv = data['last_visit']
print(f"\nLast Visit:")
print(f"   Prediction ID : {lv['prediction_id']}")
print(f"   Decision      : {lv['decision']}")
print(f"   Default Prob  : {lv['default_probability']}")
print(f"   Cluster       : {lv['cluster_label']}")
print(f"   Inputs        : {lv['inputs']}")

Status : 200

CNIC           : 42101-1234567-1
First Seen     : 2026-03-15 16:57:38.627692
Last Seen      : 2026-03-17 16:23:39.501770
Total Visits   : 13
Total Approved : 2
Total Rejected : 11
Last Decision  : REJECTED
Last Loan Amnt : 15000.0
Last Rate      : 19.4

Last Visit:
   Prediction ID : 30
   Decision      : REJECTED
   Default Prob  : 1.0
   Cluster       : High Value Borrower
   Inputs        : {'loan_amnt': 15000.0, 'loan_int_rate': 19.4, 'loan_grade': 'F', 'loan_percent_income': 0.43, 'loan_intent': 'DEBTCONSOLIDATION', 'person_income': 35000.0, 'person_age': 28, 'person_emp_length': 2.0, 'person_home_ownership': 'RENT', 'cb_person_default_on_file': 'N'}


### Step 7: Test DELETE /applicant/{cnic}

In [24]:
# First create a temporary applicant to delete
temp_payload = {
    "cnic"                      : "99999-0000000-0",
    "loan_amnt"                 : 5000,
    "loan_grade"                : "A",
    "loan_intent"               : "EDUCATION",
    "loan_percent_income"       : 0.08,
    "person_income"             : 65000,
    "person_age"                : 35,
    "person_emp_length"         : 8.0,
    "person_home_ownership"     : "MORTGAGE",
    "cb_person_default_on_file" : "N",
    "cb_person_cred_hist_length": 10.0
}

# Create
r = requests.post(f"{BASE_URL}/predict/new", json=temp_payload)
print(f"Created  : Status {r.status_code} — Prediction ID {r.json()['prediction_id']}")

# Verify exists
r = requests.get(f"{BASE_URL}/applicant/99999-0000000-0")
print(f"Exists   : Status {r.status_code} — Visits {r.json()['total_visits']}")

# Delete
r = requests.delete(f"{BASE_URL}/applicant/99999-0000000-0")
print(f"Deleted  : Status {r.status_code} — {r.json()['message']}")

# Verify gone
r = requests.get(f"{BASE_URL}/applicant/99999-0000000-0")
print(f"Verify   : Status {r.status_code} — {r.json()['detail']}")

Created  : Status 200 — Prediction ID 32
Exists   : Status 200 — Visits 1
Deleted  : Status 200 — Applicant 99999-0000000-0 and all records deleted successfully
Verify   : Status 404 — No applicant found with CNIC: 99999-0000000-0


### Step 8: Edge cases — invalid inputs

In [25]:
# Test 1 — Invalid loan grade
r = requests.post(f"{BASE_URL}/predict/new", json={
    **temp_payload, "loan_grade": "Z"
})
print(f"Invalid grade     : Status {r.status_code} (expected 422)")

# Test 2 — Invalid default flag
r = requests.post(f"{BASE_URL}/predict/new", json={
    **temp_payload, "cb_person_default_on_file": "X"
})
print(f"Invalid default   : Status {r.status_code} (expected 422)")

# Test 3 — Negative income
r = requests.post(f"{BASE_URL}/predict/new", json={
    **temp_payload, "person_income": -1000
})
print(f"Negative income   : Status {r.status_code} (expected 422)")

# Test 4 — Missing CNIC
payload_no_cnic = {k: v for k, v in temp_payload.items() if k != "cnic"}
r = requests.post(f"{BASE_URL}/predict/new", json=payload_no_cnic)
print(f"Missing CNIC      : Status {r.status_code} (expected 422)")

# Test 5 — CNIC not found
r = requests.get(f"{BASE_URL}/applicant/00000-0000000-0")
print(f"CNIC not found    : Status {r.status_code} (expected 404)")

# Test 6 — Missing loan_int_rate for existing loan
payload_no_rate = {k: v for k, v in temp_payload.items()}
r = requests.post(f"{BASE_URL}/predict/existing", json=payload_no_rate)
print(f"Missing rate      : Status {r.status_code} (expected 422)")

Invalid grade     : Status 422 (expected 422)
Invalid default   : Status 422 (expected 422)
Negative income   : Status 422 (expected 422)
Missing CNIC      : Status 422 (expected 422)
CNIC not found    : Status 404 (expected 404)
Missing rate      : Status 422 (expected 422)


### Step 9: Document Report

In [26]:
# see in docs/testing_report.md